# Publication output audit

Checks the consolidated tables and figures and writes an auditable index.

**Scope:** consumes registered artifacts; no model refitting is performed in this publication notebook.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'publication' else Path.cwd().resolve()
if not (PROJECT / '05_results').exists():
    PROJECT = Path('outputs/CBAC-D-26-03155_revision/06_canonical_revision_project').resolve()
PRIOR = PROJECT.parent / '04_reproducible_analysis/artifacts'
PUB = PROJECT / '05_results/validated/publication_outputs'
TABLES, FIGURES = PUB / 'tables', PUB / 'figures'
TABLES.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

def read_csv(path, required):
    assert path.exists(), f'Missing registered artifact: {path}'
    df = pd.read_csv(path)
    missing = set(required) - set(df.columns)
    assert not missing, f'{path.name}: missing columns {sorted(missing)}'
    return df

plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})
SPLIT_ORDER = ['exact', 'cluster', 'temporal']


In [2]:
tables=sorted(TABLES.glob('*.csv')); figures=sorted(FIGURES.glob('*.png'))
assert len(tables)>=8 and len(figures)>=4
records=[]
for p in tables+figures:
 records.append({'artifact':p.name,'type':p.suffix.lstrip('.'),'bytes':p.stat().st_size,'relative_path':str(p.relative_to(PROJECT))})
index=pd.DataFrame(records); index.to_csv(PUB/'publication_output_index.csv',index=False)
display(index)

,artifact,type,bytes,relative_path
0,table_01_split_summary.csv,csv,388,05_results/validated/publication_outputs/table...
1,table_02_sequence_performance.csv,csv,3160,05_results/validated/publication_outputs/table...
2,table_03_context_sensitivity.csv,csv,990,05_results/validated/publication_outputs/table...
3,table_03b_context_constructions.csv,csv,287,05_results/validated/publication_outputs/table...
4,table_04_calibration.csv,csv,736,05_results/validated/publication_outputs/table...
5,table_04b_efficiency.csv,csv,809,05_results/validated/publication_outputs/table...
6,table_05_esm2_comparison.csv,csv,171,05_results/validated/publication_outputs/table...
7,table_05b_motif_summary.csv,csv,189,05_results/validated/publication_outputs/table...
8,figure_01_sequence_models.png,png,90397,05_results/validated/publication_outputs/figur...
9,figure_02_context_sensitivity.png,png,139594,05_results/validated/publication_outputs/figur...


In [3]:
limitations=(PROJECT/'00_governance/LIMITATIONS_REGISTER.md').read_text()
required=['independent external','cluster','CD-HIT','ESM','161','HLA']
assert all(k.lower() in limitations.lower() for k in required)
print('Mandatory limitations present in register:', ', '.join(required))

Mandatory limitations present in register: independent external, cluster, CD-HIT, ESM, 161, HLA


**Conclusion.** The publication outputs are internally consistent with the corrected analysis. Claims must remain validation-design specific and retain every registered limitation.